# PDAC per-sample run, k_graph = 10

1. Rebuild the PDAC `.pt` files with a 10-neighbour spatial graph (new folder).
2. Adapt and encode every sample (one model per sample), same settings as before.
3. Save H&E | prototypes | z0 side by side for each sample.

Nothing in the k = 40 folders is touched.

In [2]:
import os, sys, json, glob, importlib
import numpy as np
import pandas as pd
import torch
import anndata as ad
import scanpy as sc
import matplotlib.pyplot as plt
from pathlib import Path

# --- paths ------------------------------------------------------------
SCRIPT_DIR  = "/DATA1/Roxana/Hierarchy/Two_levels/clustering"       # folder with clinical_preprocess_rawcount.py
META_H5AD   = "/DATA2/Sally/data/xenium_data/adata_xenium.h5ad"
VOCAB_PATH  = "/DATA1/Roxana/Hierarchy/Two_levels/default_gene_vocab.json"
CHECKPOINT  = "/DATA1/Roxana/Hierarchy/Two_levels/hebi_foundation_best_new.pt"
PT_DIR      = "/DATA1/Roxana/Hierarchy/xenium_clinical_raw_k10"   # new .pt files
OUT_ROOT    = Path("results_v2/pdac_k10")                         # new results
HE_DIR      = Path("/DATA2/Sally/data/xenium_data")
# --- settings (same as the k=40 run, except K_GRAPH) ------------------
K_GRAPH         = 10
N_METIS         = 1024
CANCER_TYPE     = "pdac"
LABEL_COLS      = ["response", "TRGStage"]
NUM_EPOCHS      = 15
LR              = 1e-4
SEED            = 1990
N_CELL_CLUSTERS = 10
N_NICHES        = [8, 10, 12]
RUN_GRN         = True      # set False to skip the GRN report and save time
DEVICE          = "cuda:0"
# ----------------------------------------------------------------------

os.makedirs(PT_DIR, exist_ok=True)
OUT_ROOT.mkdir(parents=True, exist_ok=True)
(OUT_ROOT / "maps").mkdir(exist_ok=True)

## 1. Preprocess: PDAC sections with k = 10 → `.pt`

In [3]:
sys.path.insert(0, SCRIPT_DIR)
cp = importlib.import_module("yULI_RAW_PREPROCESS")

full = sc.read_h5ad(META_H5AD)
pdac = full[full.obs["CancerType"].astype(str) == CANCER_TYPE].copy()
del full
print(f"{pdac.n_obs:,} PDAC cells, {pdac.obs.SampleID.nunique()} sections")

pdac = cp.normalize_counts(pdac)          # per-cell ops: same as on the full file
vocab = json.load(open(VOCAB_PATH))
meta_info = {"species": "Human", "tissue": "Xenium", "disease": "Diseased"}

for sid in sorted(pdac.obs["SampleID"].astype(str).unique()):
    out = os.path.join(PT_DIR, f"{sid}.pt")
    if os.path.exists(out):
        print(f"  {sid}: exists, skipped")
        continue
    sub = pdac[pdac.obs["SampleID"].astype(str) == sid].copy()
    d = cp.build_sample(sub, vocab, meta_info, K_GRAPH, N_METIS)
    d["sample_id"] = sid
    d["patient_id"] = str(sub.obs["PatientID"].iloc[0])
    labels = {c: str(sub.obs[c].iloc[0]) for c in LABEL_COLS if c in sub.obs}
    d.update(labels); d["labels"] = labels
    torch.save(d, out)
    print(f"  {sid}: {sub.n_obs:,} cells | edges/cell {d['edge_index'].shape[1] / sub.n_obs:.1f}")
del pdac

486,450 PDAC cells, 17 sections
🔧 Building 10-NN spatial graph (coords=(26463, 2)) ...
✅ Graph built: 26463 nodes, 301240 edges
🔧 [METIS] Creating NetworkX graph for 26463 nodes...
🔧 [METIS] Running METIS partitioning to create 1024 partitions...
✅ [METIS] METIS partitioning complete.
🔧 [METIS] Computing super-graph adjacency...
🔧 [METIS] Computing super-graph shortest paths...
✅ [METIS] Fixed partition and distance matrix created.
  2130785-C1: 26,463 cells | edges/cell 11.4
🔧 Building 10-NN spatial graph (coords=(14442, 2)) ...
✅ Graph built: 14442 nodes, 167746 edges
🔧 [METIS] Creating NetworkX graph for 14442 nodes...
🔧 [METIS] Running METIS partitioning to create 1024 partitions...
✅ [METIS] METIS partitioning complete.
🔧 [METIS] Computing super-graph adjacency...
🔧 [METIS] Computing super-graph shortest paths...
✅ [METIS] Fixed partition and distance matrix created.
  2146816-C2: 14,442 cells | edges/cell 11.6
🔧 Building 10-NN spatial graph (coords=(41045, 2)) ...
✅ Graph built: 

## 2. Sample list and AnnData per sample

In [5]:
section_files = sorted(os.path.basename(p) for p in glob.glob(os.path.join(PT_DIR, "*.pt")))
sample_ids = [f.replace(".pt", "") for f in section_files]

H5AD_DIR = OUT_ROOT / "h5ad"
H5AD_DIR.mkdir(exist_ok=True)

def adata_from_pt(pt_path):
    d = torch.load(pt_path, map_location="cpu", weights_only=False)
    X = d["X"].float().numpy()
    a = ad.AnnData(X=X)
    a.var_names = [str(g) for g in d["gene_names"]]
    a.var_names_make_unique()
    a.obsm["spatial"] = d["coords"].numpy().astype(float)
    sid = str(d["sample_id"])
    row = {"sample_id": sid, "patient_id": str(d["patient_id"])}
    row.update({c: str(d.get("labels", {}).get(c, "nan")) for c in LABEL_COLS})
    out = H5AD_DIR / f"{sid}.h5ad"
    a.write_h5ad(out)
    return str(out), row

h5ad_files, rows = [], []
for f in section_files:
    p, r = adata_from_pt(os.path.join(PT_DIR, f))
    h5ad_files.append(p); rows.append(r)
label_df = pd.DataFrame(rows)
print(label_df.to_string(index=False))

blk = ad.read_h5ad(h5ad_files[0]).X[:500]
PREPROCESS = bool(np.allclose(blk, np.round(blk)))     # False: X is already log-normalised
print("PREPROCESS =", PREPROCESS)

 sample_id patient_id response  TRGStage
2130785-C1     965740        R      CAP1
2146816-C2     D49486       NR      CAP2
2214414-C6     D28560        R      CAP0
2334564-A4     AAB746        R CAP1_MDA1
2336662-A6     D93346       NR      CAP3
2403221-A6     AAO748       NR CAP2_MDA2
2406738-C2     AAY085        R      CAP1
2406738-C3     AAY085        R CAP1_MDA1
2407966-A5     ABV534       NR      CAP2
P250014-C2     BBC829       NR CAP3_MDA2
P250014-C6     BBC829       NR CAP3_MDA2
P250572-A1     BBA636       NR CAP2_MDA2
P250572-A4     BBA636       NR CAP2_MDA2
P250734-A2     BCA213       NR CAP3_MDA2
P250734-A9     BCA213       NR CAP3_MDA2
P250857-C2     BBT131       NR CAP2_MDA2
P250857-C5     BBT131       NR CAP2_MDA2
PREPROCESS = False


## 3. One model per sample

In [5]:
from hebi.inference.pipeline import encode_sections, cluster_sections, analyse_grn
from hebi.inference.niches import assign_niches

def one_section_dir(section_file, tag):
    """Folder holding only this section, so adaptation sees one sample."""
    sub = Path("/tmp") / f"hebi_k10_{tag}"
    sub.mkdir(parents=True, exist_ok=True)
    for old in sub.glob("*.pt"):
        old.unlink()
    (sub / section_file).symlink_to(Path(PT_DIR) / section_file)
    return str(sub)

def run_one_sample(section_file, h5ad_file, sid):
    out_dir = OUT_ROOT / sid
    out_dir.mkdir(parents=True, exist_ok=True)
    work_dir = one_section_dir(section_file, sid)

    sections, ctx = encode_sections(
        checkpoint_path=CHECKPOINT, data_dir=work_dir,
        section_files=[section_file], h5ad_files=[h5ad_file],
        vocab_path=VOCAB_PATH, section_labels=[sid],
        mode="continual", init="random", num_epochs=NUM_EPOCHS, lr=LR,
        grn=True, species="Human", preprocess=PREPROCESS,
        output_dir=str(out_dir), device=DEVICE, seed=SEED,
    )
    written = cluster_sections(sections, ctx, n_cell_clusters=N_CELL_CLUSTERS,
                               cluster_method="kmeans", point_size=4)
    assign_niches(sections, ctx, work_dir, [section_file], n_niches=N_NICHES,
                  section_names=[sid], cache_dir=out_dir)

    if RUN_GRN:
        try:
            analyse_grn(sections, ctx, cluster_key="niche_k10", top_k_edges=5,
                        run_report=True, run_ccc=False)
        except Exception as exc:
            print(f"  GRN skipped: {type(exc).__name__}: {exc}")

    a = sections[0]
    niche_keys = [f"niche_k{k}" for k in N_NICHES if f"niche_k{k}" in a.obs]
    np.savez(out_dir / f"{sid}_niches.npz",
             coords=np.asarray(a.obsm["spatial"]),
             prototypes=a.obs["prototypes"].astype(int).values,
             z0_clusters=a.obs[written["cell_clusters"]].astype(int).values,
             **{k: a.obs[k].astype(int).values for k in niche_keys},
             **{f"{k}_path": a.obs[f"{k}_path"].astype(str).values for k in niche_keys
                if f"{k}_path" in a.obs})
    n_proto = a.obs["prototypes"].nunique()
    return a, ctx, n_proto

In [6]:
results = {}
for f, h5, sid in zip(section_files, h5ad_files, sample_ids):
    print(f"\n{'=' * 64}\n{sid}\n{'=' * 64}")
    try:
        a, ctx, n_proto = run_one_sample(f, h5, sid)
    except Exception as exc:
        print(f"  FAILED: {type(exc).__name__}: {exc}")
        continue
    results[sid] = {"n_cells": a.n_obs, "prototypes_used": n_proto}
    print(f"  OK {sid}: {a.n_obs:,} cells | {n_proto} prototypes")
    del a, ctx
    torch.cuda.empty_cache()

summary = pd.DataFrame(results).T
summary.to_csv(OUT_ROOT / "run_summary.csv")
print(f"\n{len(results)} of {len(sample_ids)} samples completed")
print(summary.to_string())


2130785-C1
Seed set to 1990
Loading results_v2/pdac_k10/h5ad/2130785-C1.h5ad
Covered genes: 4988/4988 (100.00%)
  26463 cells, 4988 genes
  2130785-C1.pt: 26463 cells, already aligned

Building gene-interaction graph
🧬 Building directed heterogeneous GRN (Species: Human)...
   Fetching curated priors (Pathway, LigRec, TF-Target)...
   Fetching 'pathwayextra' from OmniPath web API (whole dataset)...
   [cache-write] updated local cache for 'pathwayextra': 48764 rows
   Fetching 'ligrecextra' from OmniPath web API (whole dataset)...
   [cache-write] updated local cache for 'ligrecextra': 6658 rows
   Fetching 'dorothea' from OmniPath web API (whole dataset)...
   [cache-write] updated local cache for 'dorothea': 15267 rows
   Computing data-driven Spearman edges...
   [info] input already log-scale (max=3.90) -- skipping normalize_total/log1p to avoid double-normalizing
   Constructing PyG graph with directed edge typing...
✅ Directed GRN built! Nodes: 4988, Edges: 47148
   Pathway edge

Encoding /tmp/hebi_k10_2130785-C1/2130785-C1.pt: 100%|█| 13/13 [00:02<00:00,  5.


  26,463 cells encoded; 20/20 prototypes in use

Cell-level clustering: kmeans, k=10  ->  obs['z0_kmeans_10']

2130785-C1
  z0_kmeans_10           FIDE=0.6515
  prototypes             FIDE=0.8610

  Saved z0_kmeans_10 (s=4.0)
📦 Loading preprocessed slide from /tmp/hebi_k10_2130785-C1/2130785-C1.pt
✅ Loaded data: X=torch.Size([26463, 4988]), coords=torch.Size([26463, 2]), edge_index=torch.Size([2, 301240])


Encoding /tmp/hebi_k10_2130785-C1/2130785-C1.pt: 100%|█| 13/13 [00:02<00:00,  5.


📦 Loading preprocessed slide from /tmp/hebi_k10_2130785-C1/2130785-C1.pt
✅ Loaded data: X=torch.Size([26463, 4988]), coords=torch.Size([26463, 2]), edge_index=torch.Size([2, 301240])
niche_k8: 8 niches (2130785-C1 8) | 20 prototypes, 12 merges
niche_k10: 10 niches (2130785-C1 10) | 20 prototypes, 10 merges
niche_k12: 12 niches (2130785-C1 12) | 20 prototypes, 8 merges

GRN analysis, grouped by 'niche_k10'
  16 interactions across 9 groups -> results_v2/pdac_k10/2130785-C1/grn_top5_2130785-C1

  GRN report -> results_v2/pdac_k10/2130785-C1/2130785-C1_plot/grn
  Excluding 19403 reverse-only copies from the report

GLOBAL PRE-CLUSTERING GRN REPORT

[1] Top edges by attention
source_gene target_gene  attention edge_types  curated
      CUZD1       HOXB7   0.728616   spearman    False
      STX1A     CACNA1G   0.685937    pathway     True
      CD177       GPR12   0.928077   spearman    False
      CD177        GJB6   0.927773   spearman    False
        MYC      SNRPD1   0.436451    pathwa

Encoding /tmp/hebi_k10_2146816-C2/2146816-C2.pt: 100%|█| 8/8 [00:00<00:00,  8.08


... (60 outputs left out)

Encoding /tmp/hebi_k10_P250857-C5/P250857-C5.pt: 100%|█| 23/23 [00:03<00:00,  6.


  45,274 cells encoded; 15/20 prototypes in use

Cell-level clustering: kmeans, k=10  ->  obs['z0_kmeans_10']

P250857-C5
  z0_kmeans_10           FIDE=0.5600
  prototypes             FIDE=0.9322

  Saved z0_kmeans_10 (s=4.0)
📦 Loading preprocessed slide from /tmp/hebi_k10_P250857-C5/P250857-C5.pt
✅ Loaded data: X=torch.Size([45274, 4988]), coords=torch.Size([45274, 2]), edge_index=torch.Size([2, 508536])


Encoding /tmp/hebi_k10_P250857-C5/P250857-C5.pt: 100%|█| 23/23 [00:03<00:00,  6.


📦 Loading preprocessed slide from /tmp/hebi_k10_P250857-C5/P250857-C5.pt
✅ Loaded data: X=torch.Size([45274, 4988]), coords=torch.Size([45274, 2]), edge_index=torch.Size([2, 508536])
niche_k8: 8 niches (P250857-C5 8) | 15 prototypes, 7 merges
niche_k10: 10 niches (P250857-C5 10) | 15 prototypes, 5 merges
niche_k12: 12 niches (P250857-C5 12) | 15 prototypes, 3 merges

GRN analysis, grouped by 'niche_k10'
  15 interactions across 10 groups -> results_v2/pdac_k10/P250857-C5/grn_top5_P250857-C5

  GRN report -> results_v2/pdac_k10/P250857-C5/P250857-C5_plot/grn
  Excluding 19391 reverse-only copies from the report

GLOBAL PRE-CLUSTERING GRN REPORT

[1] Top edges by attention
source_gene target_gene  attention     edge_types  curated
       CDH1       KLRG1   0.701085         ligrec     True
      EPCAM      RNF128   0.583425       spearman    False
      EPCAM      ERRFI1   0.680046       spearman    False
      FOXA1        SOX5   0.672339      tf_target     True
      CASP1       EIF4E  

## 4. H&E | prototypes | z0

Core matching (from the k = 40 session):
- s991: H&E row = Xenium col + 1, H&E col = 9 − Xenium row (pitch 2550 µm)
- s767: H&E row = Xenium col + 1, H&E col = 8 − Xenium row (pitch 2780 µm); cores taken from the grid estimate because neighbouring cores merge in the thumbnail.

Maps are rotated 90° to match the H&E.

In [7]:
import tifffile, zarr
from PIL import Image
from scipy import ndimage
from sklearn.cluster import KMeans

HE    = {"s991": HE_DIR / "991HE.ome.tif", "s767": HE_DIR / "s767-HE.ome.tif"}
GRID  = {"s991": (4, 9), "s767": (4, 8)}
PITCH = {"s991": 2550.0, "s767": 2780.0}
NCOL  = {"s991": 9, "s767": 8}

def make_thumb(slide):
    out = OUT_ROOT / f"HE_thumb_{slide}.png"
    if not out.exists():
        with tifffile.TiffFile(HE[slide]) as t:
            s = t.series[0]
            lv = s.levels[-1] if len(s.levels) > 1 else s
            img = lv.asarray()
        if img.ndim == 3 and img.shape[0] in (3, 4):
            img = np.moveaxis(img, 0, -1)
        im = Image.fromarray(img[..., :3])
        im.thumbnail((2000, 2000))
        im.save(out)
    return out

def find_cores(slide):
    thumb = make_thumb(slide)
    im = np.asarray(Image.open(thumb).convert("L")).astype(float)
    tissue = (im < 215) & (im > 60)                     # exclude black border
    mask = ndimage.binary_fill_holes(ndimage.binary_closing(tissue, iterations=2))
    mask = ndimage.binary_opening(mask, iterations=2)
    lab, n = ndimage.label(mask)
    objs = ndimage.find_objects(lab)
    sizes = np.array([(lab[o] == i + 1).sum() for i, o in enumerate(objs)])
    keep = np.where(sizes > 0.3 * np.median(np.sort(sizes)[-20:]))[0]
    df = pd.DataFrame([dict(y0=objs[i][0].start, y1=objs[i][0].stop,
                            x0=objs[i][1].start, x1=objs[i][1].stop) for i in keep])
    df["cy"], df["cx"] = (df.y0 + df.y1) / 2, (df.x0 + df.x1) / 2
    nr, nc = GRID[slide]
    for ax_, k, col in [("cy", nr, "row"), ("cx", nc, "col")]:
        km = KMeans(k, n_init=10, random_state=0).fit(df[[ax_]])
        order = np.argsort(km.cluster_centers_.ravel())
        df[col] = pd.Series(km.labels_).map({o: i + 1 for i, o in enumerate(order)}).values
    print(f"{slide}: {n} components, {len(df)} cores kept")
    return df, im.shape

def core_box(cores, row, col, force_grid=False):
    med_w = (cores.x1 - cores.x0).median()
    med_h = (cores.y1 - cores.y0).median()
    hit = cores[(cores.row == row) & (cores.col == col)]
    if len(hit) and not force_grid:
        r = hit.iloc[0]
        if (r.x1 - r.x0) < 1.5 * med_w and (r.y1 - r.y0) < 1.5 * med_h:
            return r.y0, r.y1, r.x0, r.x1
    ok = cores[((cores.x1 - cores.x0) < 1.3 * med_w) & ((cores.y1 - cores.y0) < 1.3 * med_h)]
    cy = ok.loc[ok.row == row, "cy"].median()
    cx = ok.loc[ok.col == col, "cx"].median()
    if np.isnan(cy): cy = cores.loc[cores.row == row, "cy"].median()
    if np.isnan(cx): cx = cores.loc[cores.col == col, "cx"].median()
    w, h = (ok.x1 - ok.x0).median(), (ok.y1 - ok.y0).median()
    return cy - h / 2, cy + h / 2, cx - w / 2, cx + w / 2

def crop_core(slide, row, col, cores, thumb_shape, level=1):
    z = zarr.open(tifffile.imread(HE[slide], aszarr=True), mode="r")
    arr = z[str(level)] if hasattr(z, "keys") and str(level) in z else (z["0"] if hasattr(z, "keys") else z)
    sh = arr.shape
    H, W = (sh[0], sh[1]) if sh[-1] in (3, 4) else (sh[1], sh[2])
    sy, sx = H / thumb_shape[0], W / thumb_shape[1]
    y0, y1, x0, x1 = core_box(cores, row, col, force_grid=(slide == "s767"))
    y0, y1 = max(int(y0 * sy), 0), min(int(y1 * sy), H)
    x0, x1 = max(int(x0 * sx), 0), min(int(x1 * sx), W)
    img = arr[y0:y1, x0:x1] if sh[-1] in (3, 4) else np.moveaxis(arr[:, y0:y1, x0:x1], 0, -1)
    return np.asarray(img)[..., :3]

def he_crop_union(slide, cells, cores, thumb_shape):
    cells = sorted(cells)
    imgs = [crop_core(slide, r, c, cores, thumb_shape) for r, c in cells]
    if len(imgs) == 1:
        return imgs[0]
    if len({c for _, c in cells}) == 1:                 # same H&E column -> stack
        w = max(i.shape[1] for i in imgs)
        imgs = [np.pad(i, ((0, 0), (0, w - i.shape[1]), (0, 0)), constant_values=255) for i in imgs]
        return np.concatenate(imgs, axis=0)
    h = max(i.shape[0] for i in imgs)
    imgs = [np.pad(i, ((0, h - i.shape[0]), (0, 0), (0, 0)), constant_values=255) for i in imgs]
    return np.concatenate(imgs, axis=1)

In [9]:
def cat_scatter(ax, xy, lab, title):
    cats = pd.Categorical(lab, categories=sorted(set(lab), key=int))
    cmap = plt.get_cmap("tab20", max(len(cats.categories), 1))
    for i, c in enumerate(cats.categories):
        s = np.asarray(cats == c)
        ax.scatter(xy[s, 0], xy[s, 1], s=1.5, color=cmap(i), linewidths=0,
                   rasterized=True, label=f"{c} ({s.mean():.0%})")
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title(title, fontsize=13)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=4, fontsize=9,
              frameon=False, markerscale=6)

for sid in results:
    d = np.load(OUT_ROOT / sid / f"{sid}_niches.npz", allow_pickle=True)
    xy_rot = np.c_[-d["coords"][:, 1], d["coords"][:, 0]]        # rotate to H&E
    resp = label_df.set_index("sample_id").loc[sid, "response"]

    fig, axes = plt.subplots(1, 3, figsize=(24, 9.5))
    if sid in HE_CELLS:
        slide, cells = HE_CELLS[sid]
        cores, tshape = CORES[slide]
        axes[0].imshow(he_crop_union(slide, cells, cores, tshape))
        axes[0].set_title(f"H&E {slide} {cells}", fontsize=13)
    axes[0].axis("off")
    cat_scatter(axes[1], xy_rot, d["prototypes"].astype(str), "prototypes")
    cat_scatter(axes[2], xy_rot, d["z0_clusters"].astype(str), "z0 (k-means, 10)")
    fig.suptitle(f"{sid}   response={resp}   k_graph={K_GRAPH}", fontsize=15)

    for out in [OUT_ROOT / sid / f"{sid}_HE_proto_z0.png",
                OUT_ROOT / "maps" / f"{sid}_HE_proto_z0.png"]:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(sid, "->", OUT_ROOT / "maps" / f"{sid}_HE_proto_z0.png")

2130785-C1 -> results_v2/pdac_k10/maps/2130785-C1_HE_proto_z0.png
2146816-C2 -> results_v2/pdac_k10/maps/2146816-C2_HE_proto_z0.png
2214414-C6 -> results_v2/pdac_k10/maps/2214414-C6_HE_proto_z0.png
2334564-A4 -> results_v2/pdac_k10/maps/2334564-A4_HE_proto_z0.png
2336662-A6 -> results_v2/pdac_k10/maps/2336662-A6_HE_proto_z0.png
2403221-A6 -> results_v2/pdac_k10/maps/2403221-A6_HE_proto_z0.png
2406738-C2 -> results_v2/pdac_k10/maps/2406738-C2_HE_proto_z0.png
2406738-C3 -> results_v2/pdac_k10/maps/2406738-C3_HE_proto_z0.png
2407966-A5 -> results_v2/pdac_k10/maps/2407966-A5_HE_proto_z0.png
P250014-C2 -> results_v2/pdac_k10/maps/P250014-C2_HE_proto_z0.png
P250014-C6 -> results_v2/pdac_k10/maps/P250014-C6_HE_proto_z0.png
P250572-A1 -> results_v2/pdac_k10/maps/P250572-A1_HE_proto_z0.png
P250572-A4 -> results_v2/pdac_k10/maps/P250572-A4_HE_proto_z0.png
P250734-A2 -> results_v2/pdac_k10/maps/P250734-A2_HE_proto_z0.png
P250734-A9 -> results_v2/pdac_k10/maps/P250734-A9_HE_proto_z0.png
P250857-C2

In [10]:
from scipy import sparse
from scipy.spatial import cKDTree
from sklearn.neighbors import NearestNeighbors

MARKER_METHOD = "detection"     # "detection" = fast (pct_in - pct_out) | "wilcoxon" = as before, slow
N_TOP = 15
MIN_CELLS_DE = 10               # smaller groups get no marker list (sizes are still reported)

LINEAGE = {
    "Epithelial":  ["EPCAM", "KRT19", "KRT7", "KRT8", "KRT18", "SOX9"],
    "Tumour-assoc":["MUC1", "KRT17", "S100A2", "TFF1", "GATA6", "CEACAM5", "CYP3A5", "ITGB4"],
    "Acinar":      ["PRSS1", "CPA1", "CTRB1", "CTRL", "CUZD1", "CELA3A"],
    "Ductal":      ["CFTR", "KCNJ15", "SPP1"],
    "Endocrine":   ["CHGA", "INS", "GCG", "SCG5", "SLC30A8"],
    "Fibroblast":  ["COL1A1", "COL1A2", "DCN", "LUM", "PDGFRA"],
    "Myofibro":    ["ACTA2", "TAGLN", "POSTN"],
    "Endothelial": ["PECAM1", "VWF", "CDH5", "PLVAP"],
    "Pericyte":    ["RGS5", "CSPG4", "PDGFRB"],
    "Schwann":     ["S100B", "SOX10", "MPZ", "PLP1"],
    "T":           ["CD3D", "CD3E", "TRAC", "CD2"],
    "Cytotoxic":   ["CD8A", "CD8B", "NKG7", "GZMB", "PRF1"],
    "B":           ["MS4A1", "CD79A", "CD79B", "CD19", "CR2"],
    "Plasma":      ["JCHAIN", "MZB1"],
    "Myeloid":     ["LST1", "TYROBP", "CD68", "CSF1R", "CD163"],
}
SHORT = {"Fib": "Fib", "TCell": "T", "Epi": "Epi", "Macrophage": "Mac", "BCell": "B",
         "vEndo": "Endo", "Lymph_Endo": "LymEndo", "pericyte": "Peri", "DC": "DC",
         "Plasma": "Plasma", "Schwann_cell": "Schwann", "Neutrophil": "Neut", "Mast": "Mast",
         "SMC": "SMC", "Adipocyte": "Adipo", "low_feature": "LowFeat", "pDC": "pDC", "Neuron": "Neuron"}

mt_all  = meta.obs["maintypes"].astype(str).values
sid_all = mobs.SampleID.values

def markers_detection(D, lab, genes, groups):
    n, tot, out = D.shape[0], np.asarray(D.sum(0)).ravel(), {}
    for g in groups:
        m = lab == g
        cin = np.asarray(D[m].sum(0)).ravel()
        pin, pout = cin / m.sum(), (tot - cin) / max(n - m.sum(), 1)
        ok = (pin >= 0.05) & (pin >= 1.5 * pout)
        idx = np.where(ok)[0][np.argsort(-(pin - pout)[ok])][:N_TOP]
        out[g] = ", ".join(f"{genes[i]}({pin[i]:.0%}/{pout[i]:.0%})" for i in idx)
    return out

def markers_wilcoxon(a, lab, groups):
    keep = np.asarray((a.X > 0).mean(0)).ravel() >= 0.01
    b = a[:, keep].copy()
    b.obs["g"] = pd.Categorical(lab.astype(str))
    sc.tl.rank_genes_groups(b, "g", groups=[str(g) for g in groups], reference="rest",
                            method="wilcoxon", pts=True, key_added="rg")
    df = sc.get.rank_genes_groups_df(b, group=None, key="rg")
    df = df[df.scores > 0]
    return {int(g): ", ".join(f"{r.names}({r.pct_nz_group:.0%}/{r.pct_nz_reference:.0%})"
                              for r in df[df.group == str(g)].nlargest(N_TOP, "scores").itertuples())
            for g in groups}

def top3(values):
    s = pd.Series(values).value_counts(normalize=True).head(3)
    return ", ".join(f"{k} {v:.0%}" for k, v in s.items())

rows = []
lab_info = label_df.set_index("sample_id")
for sid, h5 in zip(sample_ids, h5ad_files):
    npz = OUT_ROOT / sid / f"{sid}_niches.npz"
    if not npz.exists():
        print(f"{sid}: no npz, skipped"); continue
    d = np.load(npz, allow_pickle=True)
    a = ad.read_h5ad(h5)
    X = sparse.csr_matrix(a.X)
    D = (X > 0).astype(np.float32).tocsr()
    genes = np.asarray(a.var_names)
    gidx = {g: i for i, g in enumerate(genes)}
    coords = d["coords"]

    m = sid_all == sid
    dist, ix = cKDTree(mxy[m]).query(coords)
    mt = np.array([SHORT.get(t, t) for t in mt_all[m][ix]])
    if (dist > 0.5).mean() > 0.01:
        print(f"  [warn] {sid}: {(dist > 0.5).mean():.1%} cells not matched exactly to maintypes")

    nn = NearestNeighbors(n_neighbors=11).fit(coords).kneighbors(return_distance=False)[:, 1:]

    for level, key, other_key, other_name in [("prototype", "prototypes", "z0_clusters", "z0"),
                                              ("z0", "z0_clusters", "prototypes", "proto")]:
        lab, other = d[key].astype(int), d[other_key].astype(int)
        groups = sorted(np.unique(lab))
        de_groups = [g for g in groups if (lab == g).sum() >= MIN_CELLS_DE]
        mk = (markers_wilcoxon(a, lab, de_groups) if MARKER_METHOD == "wilcoxon"
              else markers_detection(D, lab, genes, de_groups))
        for g in groups:
            sel = lab == g
            row = {"sample_id": sid, "patient_id": lab_info.loc[sid, "patient_id"],
                   "response": lab_info.loc[sid, "response"], "level": level, "cluster": g,
                   "n_cells": int(sel.sum()), "pct_cells": round(100 * sel.mean(), 2),
                   "cohesion": round(float((lab[nn[sel]] == g).mean()), 2)}
            Dm = D[sel]
            for lin, gl in LINEAGE.items():
                gi = [gidx[x] for x in gl if x in gidx]
                row[lin] = int(round(100 * (np.asarray(Dm[:, gi].sum(1)).ravel() > 0).mean())) if gi else None
            row["maintypes_top3"] = top3(mt[sel])
            row[f"{other_name}_top3"] = top3([f"{other_name} {x}" for x in other[sel]])
            row["markers"] = mk.get(g, "(too few cells)")
            rows.append(row)
    print(f"{sid}: done")
    del a, X, D

ann_in = pd.DataFrame(rows)
ann_in.to_csv(OUT_ROOT / "annotation_input_all.csv", index=False)
print(ann_in.shape, "->", OUT_ROOT / "annotation_input_all.csv")

2130785-C1: done
2146816-C2: done
2214414-C6: done
2334564-A4: done
2336662-A6: done
2403221-A6: done
2406738-C2: done
2406738-C3: done
2407966-A5: done
P250014-C2: done
P250014-C6: done
P250572-A1: done
P250572-A4: done
P250734-A2: done
P250734-A9: done
P250857-C2: done
P250857-C5: done
(474, 27) -> results_v2/pdac_k10/annotation_input_all.csv


In [17]:
SHOW_PCT = False   # True: add each label's % of cells to the legend

from matplotlib.lines import Line2D
ANN = pd.read_csv(OUT_ROOT / "HEBI_pdac_k10_annotation_final_v3.csv")
(OUT_ROOT / "final").mkdir(exist_ok=True)

def ann_scatter(ax, xy, ids, table, title):
    hs = []
    for r in table.itertuples():
        s = ids == int(r.cluster)
        if not s.any():
            continue
        ax.scatter(xy[s, 0], xy[s, 1], s=2.5, color=r.colour, linewidths=0, rasterized=True)
        hs.append(Line2D([], [], marker="o", ls="", color=r.colour, ms=7,
                         label=f"{r.plot_label} ({s.mean():.1%})" if SHOW_PCT else r.plot_label))
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_title(title, fontsize=13)
    ax.legend(handles=hs, loc="upper center", bbox_to_anchor=(0.5, -0.02), ncol=1,
              fontsize=8.5, frameon=False)

for sid in sample_ids:
    f = OUT_ROOT / sid / f"{sid}_niches.npz"
    if not f.exists():
        continue
    d = np.load(f, allow_pickle=True)
    xy = np.c_[-d["coords"][:, 1], d["coords"][:, 0]]
    a = ANN[ANN.sample_id == sid]

    # niches: prototypes with the same label share one colour and one legend entry
    pt = a[a.level == "prototype"]
    nid = pt.set_index("cluster")["niche_id"].astype(int).to_dict()
    niche_ids = np.array([nid.get(int(p), -1) for p in d["prototypes"]])
    ntab = pt.drop_duplicates("niche_id").assign(cluster=lambda x: x.niche_id.astype(int)).sort_values("cluster")
    ntab = pd.concat([ntab, pd.DataFrame([{"cluster": -1, "plot_label": "prototype <10 cells", "colour": "#c7c7c7"}])])

    # z0: every cluster separate
    ztab = a[a.level == "z0"].sort_values("cluster")
    z0 = d["z0_clusters"].astype(int)

    fig, axes = plt.subplots(1, 3, figsize=(28, 12))
    if sid in HE_CELLS:
        slide, cells = HE_CELLS[sid]
        cores, tshape = CORES[slide]
        axes[0].imshow(he_crop_union(slide, cells, cores, tshape))
        axes[0].set_title("H&E", fontsize=13)
    axes[0].axis("off")
    ann_scatter(axes[1], xy, niche_ids, ntab, f"Niches ({pt.niche_id.nunique()})")
    ann_scatter(axes[2], xy, z0, ztab, "z0 (10 clusters)")
    fig.suptitle(f"{sid}   response={label_df.set_index('sample_id').loc[sid, 'response']}", fontsize=15)
    for out in [OUT_ROOT / sid / f"{sid}_HE_niche_z0_annotated_v5.png",
                OUT_ROOT / "final" / f"{sid}_HE_niche_z0_annotated_v5.png"]:
        fig.savefig(out, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(sid, "->", out)

2130785-C1 -> results_v2/pdac_k10/final/2130785-C1_HE_niche_z0_annotated_v5.png
2146816-C2 -> results_v2/pdac_k10/final/2146816-C2_HE_niche_z0_annotated_v5.png
2214414-C6 -> results_v2/pdac_k10/final/2214414-C6_HE_niche_z0_annotated_v5.png
2334564-A4 -> results_v2/pdac_k10/final/2334564-A4_HE_niche_z0_annotated_v5.png
2336662-A6 -> results_v2/pdac_k10/final/2336662-A6_HE_niche_z0_annotated_v5.png
2403221-A6 -> results_v2/pdac_k10/final/2403221-A6_HE_niche_z0_annotated_v5.png
2406738-C2 -> results_v2/pdac_k10/final/2406738-C2_HE_niche_z0_annotated_v5.png
2406738-C3 -> results_v2/pdac_k10/final/2406738-C3_HE_niche_z0_annotated_v5.png
2407966-A5 -> results_v2/pdac_k10/final/2407966-A5_HE_niche_z0_annotated_v5.png
P250014-C2 -> results_v2/pdac_k10/final/P250014-C2_HE_niche_z0_annotated_v5.png
P250014-C6 -> results_v2/pdac_k10/final/P250014-C6_HE_niche_z0_annotated_v5.png
P250572-A1 -> results_v2/pdac_k10/final/P250572-A1_HE_niche_z0_annotated_v5.png
P250572-A4 -> results_v2/pdac_k10/final/

In [18]:
# ---- Individual plots per sample: H&E | niches | z0 as 3 separate SVGs

import matplotlib as mpl
from matplotlib.lines import Line2D
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 14, "svg.fonttype": "none"})

ANN = pd.read_csv(OUT_ROOT / "HEBI_pdac_k10_annotation_final_v3.csv")
_t = plt.get_cmap("tab20").colors
TAB = [_t[i] for i in list(range(0, 20, 2)) + list(range(1, 20, 2)) if i != 15]  # 10 strong (tab10) colours first; light grey kept for unresolved
GREY = "#c7c7c7"
PT = 3          # point size

def map_plot(xy, ids, table, out):
    """table: one row per label with columns cluster (id in `ids`), name, colour."""
    fig, ax = plt.subplots(figsize=(8, 8))
    hs = []
    for r in table.itertuples():
        m = ids == r.cluster
        if not m.any():
            continue
        ax.scatter(xy[m, 0], xy[m, 1], s=PT, color=r.colour, linewidths=0, rasterized=True)
        hs.append(Line2D([], [], marker="o", ls="", color=r.colour, ms=9, label=r.name))
    ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.legend(handles=hs, loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
    fig.savefig(out, format="svg", dpi=300, bbox_inches="tight"); plt.close(fig)

for sid in sample_ids:
    f = OUT_ROOT / sid / f"{sid}_niches.npz"
    if not f.exists():
        continue
    out_dir = OUT_ROOT / sid / "individual_plot"; out_dir.mkdir(exist_ok=True)
    d = np.load(f, allow_pickle=True)
    xy = np.c_[-d["coords"][:, 1], d["coords"][:, 0]]          # same rotation as the H&E
    a = ANN[ANN.sample_id == sid]

    # 1. H&E
    if sid in HE_CELLS:
        slide, cells = HE_CELLS[sid]
        cores, tshape = CORES[slide]
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.imshow(he_crop_union(slide, cells, cores, tshape)); ax.axis("off")
        fig.savefig(out_dir / f"{sid}_HE.svg", format="svg", dpi=300, bbox_inches="tight"); plt.close(fig)

    # 2. niches (prototypes merged by label), tab20, unresolved in grey
    pt = a[a.level == "prototype"]
    nid = pt.set_index("cluster")["niche_id"].astype(int).to_dict()
    niche_ids = np.array([nid.get(int(p), -1) for p in d["prototypes"]])
    unres = set(pt.loc[pt.niche.str.startswith("Unresolved (minor)"), "niche_id"].astype(int))
    niche_ids = np.where(np.isin(niche_ids, list(unres)), -1, niche_ids)     # tiny unresolved prototypes -> one grey entry
    nt = pt[~pt.niche_id.astype(int).isin(unres)].drop_duplicates("niche_id").sort_values("niche_id")
    rows = [dict(cluster=int(r.niche_id), name=r.niche, colour=TAB[k % len(TAB)]) for k, r in enumerate(nt.itertuples())]
    if (niche_ids == -1).any():
        rows.append(dict(cluster=-1, name="Unresolved (minor)", colour=GREY))
    map_plot(xy, niche_ids, pd.DataFrame(rows), out_dir / f"{sid}_niches.svg")

    # 3. z0: 10 clusters, tab20 (first 10 = the strong, non-paired colours)
    zt = a[a.level == "z0"].sort_values("cluster")
    ztab = pd.DataFrame(dict(cluster=zt.cluster.astype(int).values, name=zt.plot_label.values,
                             colour=[TAB[i % len(TAB)] for i in range(len(zt))]))
    map_plot(xy, d["z0_clusters"].astype(int), ztab, out_dir / f"{sid}_z0.svg")
    print(sid, "->", out_dir)

2130785-C1 -> results_v2/pdac_k10/2130785-C1/individual_plot
2146816-C2 -> results_v2/pdac_k10/2146816-C2/individual_plot
2214414-C6 -> results_v2/pdac_k10/2214414-C6/individual_plot
2334564-A4 -> results_v2/pdac_k10/2334564-A4/individual_plot
2336662-A6 -> results_v2/pdac_k10/2336662-A6/individual_plot
2403221-A6 -> results_v2/pdac_k10/2403221-A6/individual_plot
2406738-C2 -> results_v2/pdac_k10/2406738-C2/individual_plot
2406738-C3 -> results_v2/pdac_k10/2406738-C3/individual_plot
2407966-A5 -> results_v2/pdac_k10/2407966-A5/individual_plot
P250014-C2 -> results_v2/pdac_k10/P250014-C2/individual_plot
P250014-C6 -> results_v2/pdac_k10/P250014-C6/individual_plot
P250572-A1 -> results_v2/pdac_k10/P250572-A1/individual_plot
P250572-A4 -> results_v2/pdac_k10/P250572-A4/individual_plot
P250734-A2 -> results_v2/pdac_k10/P250734-A2/individual_plot
P250734-A9 -> results_v2/pdac_k10/P250734-A9/individual_plot
P250857-C2 -> results_v2/pdac_k10/P250857-C2/individual_plot
P250857-C5 -> results_v2

In [36]:
import json, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.patches import Rectangle
from scipy.stats import mannwhitneyu
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 10, "svg.fonttype": "path",
                     "axes.spines.top": False, "axes.spines.right": False})
OUT_ROOT = Path("results_v2/pdac_k10")
FIG = OUT_ROOT / "figure_pdac"; FIG.mkdir(parents=True, exist_ok=True)
ANN = pd.read_csv(OUT_ROOT / "HEBI_pdac_k10_annotation_final_v3.csv")
PAL = json.load(open(OUT_ROOT / "hebi_pdac_palette_v3.json"))
NICHE_COL, Z0_COL = PAL["niche"], PAL["z0_class"]
A = pd.read_csv(OUT_ROOT / "rnr_analysis" / "A_class_placement_per_section.csv")
B = pd.read_csv(OUT_ROOT / "rnr_analysis" / "B_stroma_programmes_per_section.csv")
SHOW_P = True
PNG_DPI = 600                                    # every panel is also saved as PNG (BioRender drops images embedded in SVG)
def save(fig, name):
    fig.savefig(FIG / f"{name}.svg", bbox_inches="tight", dpi=300)
    fig.savefig(FIG / f"{name}.png", bbox_inches="tight", dpi=PNG_DPI, facecolor="white")
    plt.close(fig)                                   # P values on f/g panels (exact two-sided MWU, patient level)
CR, CNR = "#1f77b4", "#d62728"

meta = ANN.drop_duplicates("sample_id").set_index("sample_id")[["patient_id", "response"]]
pat = meta.drop_duplicates("patient_id").sort_values(["response", "patient_id"], ascending=[False, True])
PLAB = {p: f"{r}{i + 1}" for r in ["R", "NR"] for i, p in enumerate(pat[pat.response == r].patient_id)}
ORDER = [p for p in pat.patient_id if PLAB[p].startswith("R") and not PLAB[p].startswith("NR")] + \
        [p for p in pat.patient_id if PLAB[p].startswith("NR")]
pd.Series(PLAB, name="label").to_csv(FIG / "patient_labels.csv")      # R1.. / NR1.. mapping
nR = sum(not PLAB[p].startswith("NR") for p in ORDER)

def niche_group(n):                                                   # tiny unresolved prototypes -> one label
    return "Unresolved (minor)" if str(n).startswith("Unresolved (minor)") else n
def patient_comp(level, col):
    d = ANN[ANN.level == level].copy(); d["g"] = d[col].map(niche_group)
    s = d.groupby(["sample_id", "g"]).pct_cells.sum().unstack().fillna(0)
    return s.join(meta).groupby("patient_id").mean(numeric_only=True).reindex(ORDER)
NICHE_ORDER = [k for k in NICHE_COL if k != "Unresolved (minor)"] + ["Unresolved (minor)"]
Z0_ORDER = list(Z0_COL)
Pn = patient_comp("prototype", "niche"); Pn = Pn[[c for c in NICHE_ORDER if c in Pn]]
Pz = patient_comp("z0", "label");        Pz = Pz[[c for c in Z0_ORDER if c in Pz]]

In [37]:
# a. niche presence heatmap (patients x niche labels, % cells)
fig, ax = plt.subplots(figsize=(5.2, 5.6))
M = Pn.drop(columns=["Unresolved (minor)"], errors="ignore").T
im = ax.pcolormesh(np.arange(M.shape[1] + 1) - 0.5, np.arange(M.shape[0] + 1) - 0.5, M.values, cmap="Greys",
                   vmin=0, vmax=min(40, M.values.max()), edgecolors="white", linewidth=0.3)
ax.set_xlim(-0.5, M.shape[1] - 0.5); ax.set_ylim(M.shape[0] - 0.5, -0.5)
ax.set_xticks(range(M.shape[1]), [PLAB[p] for p in M.columns], rotation=90)
ax.set_yticks(range(M.shape[0]), M.index, fontsize=8)
ax.axvline(nR - 0.5, color="k", lw=1.2)
for i, (lab, c) in enumerate(zip(["R", "NR"], [CR, CNR])):
    ax.text([nR / 2 - 0.5, nR + (M.shape[1] - nR) / 2 - 0.5][i], -0.9, lab, color=c, ha="center", weight="bold")
for y in range(M.shape[0]):                                           # colour key = niche colour
    ax.add_patch(Rectangle((-0.95, y - 0.4), 0.4, 0.8, color=NICHE_COL[M.index[y]], clip_on=False))
cb = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02); cb.set_label("% of cells"); cb.solids.set_rasterized(False)
ax.spines[["left", "bottom"]].set_visible(False); ax.tick_params(length=0); ax.tick_params(axis="y", pad=16)
save(fig, "panel_a_niche_heatmap")

In [38]:
# b, c. stacked composition per patient
def stacked(P, colours, fname, legend_title, ylabel):
    fig, ax = plt.subplots(figsize=(5.2, 3.8))
    x = np.arange(len(P)) + (np.arange(len(P)) >= nR) * 0.6
    bottom = np.zeros(len(P))
    for c in P.columns:
        v = P[c].values / 100
        ax.bar(x, v, bottom=bottom, color=colours[c], width=0.85, label=c, lw=0); bottom += v
    ax.set_xticks(x, [PLAB[p] for p in P.index], rotation=90); ax.set_ylabel(ylabel); ax.set_ylim(0, 1.0)
    ax.axvline(nR - 0.2, color="k", ls="--", lw=0.8)
    ax.text(x[:nR].mean(), 1.03, "R", color=CR, ha="center", weight="bold")
    ax.text(x[nR:].mean(), 1.03, "NR", color=CNR, ha="center", weight="bold")
    h, l = ax.get_legend_handles_labels()
    ax.legend(h[::-1], l[::-1], title=legend_title, bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=7,
              title_fontsize=8, frameon=False)
    save(fig, fname)
stacked(Pn, NICHE_COL, "panel_b_niche_composition", "Niche", "Composition fraction")
stacked(Pz, Z0_COL, "panel_c_cell_composition", "Cell class", "Composition fraction")

In [39]:
# d, e. niche and z0 maps of two sections, with zooms (colours = cohort colours from the CSV)
SECTIONS = {"R": "2214414-C6", "NR": "P250734-A2"}
ZOOM_TARGET = {"2214414-C6": "TLS B-cell follicle", "P250734-A2": "Nerve niche"}   # zoom centred on this niche
ZOOM_UM = 600            # zoom window side (µm)
PT, PTZ = 1.2, 6         # point size full map / zoom

def load_maps(sid):
    d = np.load(OUT_ROOT / sid / f"{sid}_niches.npz", allow_pickle=True)
    xy = np.c_[-d["coords"][:, 1], d["coords"][:, 0]]                 # same rotation as the H&E
    a = ANN[ANN.sample_id == sid]
    pt = a[a.level == "prototype"].set_index("cluster")
    zt = a[a.level == "z0"].set_index("cluster")
    p = d["prototypes"].astype(int); z = d["z0_clusters"].astype(int)
    niche = np.array([niche_group(pt.niche.get(i, "Unresolved (minor)")) for i in p], dtype=object)
    ncol = np.array([NICHE_COL[n] for n in niche], dtype=object)
    zlab = np.array([zt.short_label.get(i, "Unassigned") for i in z], dtype=object)
    zcol = np.array([zt.colour.get(i, "#e6e6e6") for i in z], dtype=object)
    return xy, niche, ncol, zlab, zcol

def zoom_window(xy, niche, target, size=ZOOM_UM):
    """centre on the densest 200 µm tile of the target niche"""
    m = niche == target
    if m.sum() == 0:
        c = np.median(xy, 0)
    else:
        t = np.floor(xy[m] / 200).astype(int); k, cnt = np.unique(t, axis=0, return_counts=True)
        c = (k[cnt.argmax()] + 0.5) * 200
    return c[0] - size / 2, c[1] - size / 2, size

def map_ax(ax, xy, col, win=None, s=PT):
    if win is not None:
        x0, y0, w = win; m = (xy[:, 0] >= x0) & (xy[:, 0] <= x0 + w) & (xy[:, 1] >= y0) & (xy[:, 1] <= y0 + w)
        ax.scatter(xy[m, 0], xy[m, 1], c=list(col[m]), s=s, linewidths=0, rasterized=True)
        ax.set_xlim(x0, x0 + w); ax.set_ylim(y0 + w, y0)
        for sp in ax.spines.values(): sp.set_visible(True); sp.set_color("#d62728"); sp.set_linewidth(1.5)
        ax.set_xticks([]); ax.set_yticks([])
    else:
        ax.scatter(xy[:, 0], xy[:, 1], c=list(col), s=s, linewidths=0, rasterized=True)
        ax.set_aspect("equal"); ax.invert_yaxis(); ax.axis("off")
    ax.set_aspect("equal")

def legend_for(ax, labels, cols, order=None):
    seen = {}
    for l, c in zip(labels, cols): seen.setdefault(l, c)
    keys = [k for k in (order or seen) if k in seen]
    ax.legend(handles=[plt.Line2D([], [], marker="o", ls="", color=seen[k], ms=6, label=k) for k in keys],
              loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False, fontsize=7)
# fixed layout in inches (no tight_layout), so rows and gaps are identical in d and e
S, CG, RG, TOP, LEG = 2.4, 0.25, 0.55, 0.35, 2.6   # map size, column gap, row gap, title space, legend width
def grid_axes(fig, ncols, W, H):
    axs = []
    for r in range(2):
        y = H - TOP - S - r * (S + RG + TOP)
        axs.append([fig.add_axes([(c * (S + CG)) / W, y / H, S / W, S / H]) for c in range(ncols)])
    return axs

for panel, level in [("d", "niche"), ("e", "z0")]:
    ncols = 3 if panel == "d" else 2                  # d: H&E | niches | zoom;  e: z0 | zoom
    W = ncols * S + (ncols - 1) * CG + LEG
    H = 2 * (S + TOP) + RG
    fig = plt.figure(figsize=(W, H))
    axes = grid_axes(fig, ncols, W, H)
    for r, (resp, sid) in enumerate(SECTIONS.items()):
        xy, niche, ncol, zlab, zcol = load_maps(sid)
        win = zoom_window(xy, niche, ZOOM_TARGET[sid])
        lab, col = (niche, ncol) if level == "niche" else (zlab, zcol)
        row = list(axes[r])
        if panel == "d":
            a0 = row.pop(0)
            if sid in HE_CELLS:
                slide, cells = HE_CELLS[sid]; cores, tshape = CORES[slide]
                a0.imshow(he_crop_union(slide, cells, cores, tshape))
            a0.axis("off"); a0.set_title(f"{resp}  H&E", fontsize=9)
        a1, a2 = row
        map_ax(a1, xy, col); a1.set_title(f"{resp}  {'niches' if level == 'niche' else 'cell clusters (z0)'}", fontsize=9)
        x0, y0, w = win; a1.add_patch(Rectangle((x0, y0), w, w, fill=False, ec="#d62728", lw=1.2))
        map_ax(a2, xy, col, win=win, s=PTZ)
        inwin = (xy[:, 0] >= x0) & (xy[:, 0] <= x0 + w) & (xy[:, 1] >= y0) & (xy[:, 1] <= y0 + w)
        legend_for(a2, lab[inwin], col[inwin], NICHE_ORDER if level == "niche" else None)
    name = f"panel_{panel}_{'niche' if level == 'niche' else 'cell'}_maps"
    fig.savefig(FIG / f"{name}.svg", dpi=300)                        # no bbox_inches="tight": keep the fixed geometry
    fig.savefig(FIG / f"{name}.png", dpi=PNG_DPI, facecolor="white"); plt.close(fig)

In [40]:
# f, g. patient-level dot plots, R vs NR
def pat_mean(df, value):
    return df.join(meta, on="sample_id").groupby(["patient_id", "response"])[value].mean().dropna()

def niche_pct(labels):
    s = ANN[(ANN.level == "prototype") & ANN.niche.isin(labels)].groupby("sample_id").pct_cells.sum()
    return pd.DataFrame({"v": s.reindex(meta.index).fillna(0)}).rename_axis("sample_id").reset_index()
def z0_pct(labels):
    s = ANN[(ANN.level == "z0") & ANN.label.isin(labels)].groupby("sample_id").pct_cells.sum()
    return pd.DataFrame({"v": s.reindex(meta.index).fillna(0)}).rename_axis("sample_id").reset_index()
LYM = ["TLS B-cell follicle", "TLS follicle border", "TLS T-cell zone", "Lymphoid aggregate", "Lymphoid-enriched mixed niche"]
Tm = A[A["class_group"] == "T cell (marker)"][["sample_id", "frac_in::Lymphoid (organised)"]].fillna(0)
Tm["v"] = Tm["frac_in::Lymphoid (organised)"] * 100

PANELS = {
 "f": [("Tumour epithelial niche", niche_pct(["Tumour epithelial niche"]), "% of cells"),
       ("Desmoplastic myCAF niche", niche_pct(["Desmoplastic stromal niche (myCAF)"]), "% of cells"),
       ("Organised lymphoid niches", niche_pct(LYM), "% of cells"),
       ("T cells in organised\nlymphoid niches", Tm[["sample_id", "v"]], "% of T cells")],
 "g": [("B-containing clusters", z0_pct(["B cell", "Lymphoid (B/T mixed)"]), "% of cells"),
       ("Myeloid clusters", z0_pct(["Myeloid (macrophage/DC)", "Myeloid (macrophage/neutrophil)"]), "% of cells"),
       ("CAF classes", z0_pct(["CAF (myCAF)", "CAF (activated)", "CAF/fibroblast"]), "% of cells"),
       ("Matrix-remodelling score,\nstroma > 200 µm from tumour", B[["sample_id", "matrix::stroma_distal"]].rename(columns={"matrix::stroma_distal": "v"}), "Mean expression")],
}
rng = np.random.default_rng(0)
for panel, items in PANELS.items():
    fig, axes = plt.subplots(2, 2, figsize=(4.8, 4.8))
    for ax, (title, df, ylab) in zip(axes.ravel(), items):
        p = pat_mean(df.set_index("sample_id").reset_index(), "v")
        r, n = p.xs("R", level="response"), p.xs("NR", level="response")
        for i, (v, c) in enumerate([(r, CR), (n, CNR)]):
            ax.scatter(np.full(len(v), i) + rng.uniform(-0.1, 0.1, len(v)), v, s=22, color=c, zorder=3)
            ax.hlines(v.median(), i - 0.25, i + 0.25, color="k", lw=1.5)
        ax.set_xticks([0, 1], ["R", "NR"]); ax.set_xlim(-0.6, 1.6); ax.set_ylabel(ylab, fontsize=8)
        t = title + (f"\nP = {mannwhitneyu(r, n, method='exact').pvalue:.3f}" if SHOW_P and len(r) and len(n) else "")
        ax.set_title(t, fontsize=8)
    fig.tight_layout(); save(fig, f"panel_{panel}_by_response")
print("panels ->", FIG)

panels -> results_v2/pdac_k10/figure_pdac


## GRN

In [30]:
# %% GRN stage on metacells (production): all PDAC sections x seeds, outputs in the same format as the pipeline.
# Per section: trained gene embeddings of the main run (hebi_continual.pt, fixed), z0 classes, 500 HVG,
# graph on those HVG (prior + Spearman 0.4, self-loops removed), within-z0 metacells (k=30, all cells),

import contextlib, io, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F, scipy.sparse as sp
from pathlib import Path
from scipy.stats import spearmanr
from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from hebi.model.grn_layer import GRNRefinementLayer
from hebi.training.pretrain import Train
from hebi.grn.build import build_hetero_grn
from hebi.grn.report import run_global_grn_report
from hebi.grn.niche_activity import niche_edge_activity
from hebi.inference.pipeline import load_sections, align_sections

ROOT = Path("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10")
OUT = ROOT / "grn_metacell_persection"
ANN_CSV = "/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10/HEBI_pdac_k10_annotation_final_v3.csv"
SEEDS = [1990]# 1990 = main result; others = stability (supplement)
SIDS = sorted(p.stem for p in Path(PT_DIR).glob("*.pt"))

K, N_HVG, SPEARMAN_THR = 30, 500, 0.4
HVG_MODE = "per_section"                             # "common" (same 500 genes in every section) or "per_section"
STEPS, LR, MASK_FRAC = 3000, 1e-3, 0.15
MAKE_PLOTS_SEED = 1990                          # report figures only for the main seed
dev = torch.device(DEVICE)
OUT.mkdir(exist_ok=True)
ann = pd.read_csv(ANN_CSV)
print(f"{len(SIDS)} sections, seeds {SEEDS}, HVG mode {HVG_MODE}")

def load_ad(sid):
    return align_sections(load_sections([str(H5AD_DIR / f"{sid}.h5ad")], VOCAB_PATH, preprocess=PREPROCESS,
                                        species="Human"), PT_DIR, [f"{sid}.pt"])[0]

def gene_var(X):
    X = X.toarray() if hasattr(X, "toarray") else np.asarray(X)
    return X.var(0)

# ---------- HVG ----------
if HVG_MODE == "common":
    ranks, GENES = [], None
    for sid in SIDS:
        ad = load_ad(sid)
        GENES = list(ad.var_names) if GENES is None else GENES
        assert list(ad.var_names) == GENES, f"{sid}: gene order differs between sections"
        ranks.append(pd.Series(gene_var(ad.X)).rank(ascending=False).values)
        del ad
    HV_COMMON = np.sort(np.argsort(np.mean(ranks, axis=0))[:N_HVG])
    pd.Series([GENES[i] for i in HV_COMMON]).to_csv(OUT / "hvg_common.csv", index=False, header=["gene"])
    print(f"common HVG: {len(HV_COMMON)} genes -> {OUT / 'hvg_common.csv'}")

def drop_self_loops(gg):
    d = gg["global"]; ei = d.edge_index; m = ei[0] != ei[1]
    if bool(m.all()): return 0
    E = ei.size(1)
    keys = d.keys() if callable(d.keys) else d.keys
    for key in list(keys):
        v = d[key]
        if not torch.is_tensor(v): continue
        if key == "edge_index": d[key] = v[:, m]
        elif v.dim() >= 1 and v.size(0) == E: d[key] = v[m]
    return int((~m).sum())

LOG = []
for sid in SIDS:
    t0 = time.time()
    try:
        # ----- data, embeddings, classes -----
        ad = load_ad(sid); names = list(ad.var_names)
        work_dir = one_section_dir(f"{sid}.pt", f"{sid}_grnmeta")
        tr = Train(data_dir=work_dir, vocab_path=VOCAB_PATH, device=DEVICE, seed=0, verbose=0)
        tr.discover_data_files(); tr.data_path = tr.data_files[0]; tr.load_data()
        state = torch.load(OUT_ROOT / sid / "hebi_continual.pt", map_location=dev)
        tr.gene_embedder.load_state_dict(state["gene_embedder"]); tr.gene_linear.load_state_dict(state["gene_linear"])
        with torch.no_grad():
            GE = F.normalize(tr.gene_linear(tr.gene_embedder(tr.gene_ids.to(dev))), dim=1, p=2)
        X = tr.X.float().cpu().numpy(); N, G = X.shape
        assert ad.n_obs == N and len(names) == G
        npz = np.load(OUT_ROOT / sid / f"{sid}_niches.npz", allow_pickle=True)
        z0 = npz["z0_clusters"].astype(int); proto = npz["prototypes"].astype(int)
        assert len(z0) == N and len(proto) == N
        pt = ann[(ann.sample_id == sid) & (ann.level == "prototype")].set_index("cluster").niche
        niche = np.array([pt.get(int(p), "Unresolved (minor)") for p in proto], dtype=object)

        # ----- HVG, PCA, graph on HVG (all cells) -----
        HV = HV_COMMON if HVG_MODE == "common" else np.sort(np.argsort(-X.var(0))[:N_HVG])
        mu0, sd0 = X.mean(0), X.std(0) + 1e-6
        pca = PCA(30, random_state=0).fit((X[np.random.default_rng(0).permutation(N)[:50000]] - mu0) / sd0)
        PC = np.vstack([pca.transform((X[i:i + 8192] - mu0) / sd0) for i in range(0, N, 8192)])
        ad_h = ad[:, HV].copy(); ad_h.obs["niche"] = niche
        buf = io.StringIO()
        with contextlib.redirect_stdout(buf):
            gg, gn_h, _ = build_hetero_grn(adata=ad_h, species="Human", spearman_threshold=SPEARMAN_THR,
                                           add_reverse_for_curated=True)
        assert list(gn_h) == [names[i] for i in HV], "gene order mismatch"
        n_self = drop_self_loops(gg)
        EI = gg["global"].edge_index.to(dev); SRC, DST = EI
        n_sp = int((gg["global"].edge_attr[:, 1] > 0).sum())
        Gh = len(HV); GEh = GE[torch.as_tensor(HV, device=dev)]
        indeg = torch.zeros(Gh, device=dev).index_add_(0, DST, torch.ones_like(DST, dtype=torch.float))
        HAS_REG = indeg > 0

        # ----- within-z0 metacells over all cells -----
        Ms = []
        for g in np.unique(z0):
            sub = np.where(z0 == g)[0]
            n_c = max(1, len(sub) // K)
            lab = (np.zeros(len(sub), dtype=int) if n_c == 1 else
                   MiniBatchKMeans(n_clusters=n_c, random_state=0, batch_size=4096, n_init=3).fit(PC[sub]).labels_)
            A = sp.csr_matrix((np.ones(len(sub)), (lab, np.arange(len(sub)))), shape=(lab.max() + 1, len(sub)))
            cnt = np.asarray(A.sum(1)).ravel(); A = A[cnt > 0]; cnt = cnt[cnt > 0]
            Ms.append(np.log1p(A @ np.expm1(X[sub][:, HV]) / cnt[:, None]))
        M = np.vstack(Ms).astype(np.float32)
        mu, sd = M.mean(0), M.std(0); sd = np.where(sd > 1e-4, sd, 1.0)
        Z = torch.tensor((M - mu) / sd, device=dev)
        del X, PC, Ms

        # ----- train the GRN layer, one run per seed -----
        for seed in SEEDS:
            torch.manual_seed(seed)
            layer = GRNRefinementLayer(dim_model=GEh.shape[1], heads=4).to(dev)
            dec = nn.Linear(GEh.shape[1], 1).to(dev)
            with torch.no_grad(): layer.gat.att.zero_()                       # attention starts uniform
            params = list(layer.parameters()) + list(dec.parameters())
            opt = torch.optim.Adam(params, lr=LR)
            layer.eval(); _, a0 = layer(GEh, EI, return_attention=True); layer.train()
            losses = []
            for step in range(STEPS):
                refined, att = layer(gene_emb=GEh, edge_index_grn=EI, return_attention=True)
                agg = torch.zeros_like(Z).index_add_(1, DST, Z[:, SRC] * att.unsqueeze(0))
                pred = agg * dec(refined).squeeze(-1).unsqueeze(0)
                score = (torch.rand(Gh, device=dev) < MASK_FRAC) & HAS_REG
                loss = F.mse_loss(pred[:, score], Z[:, score])
                opt.zero_grad(set_to_none=True); loss.backward()
                torch.nn.utils.clip_grad_norm_(params, max_norm=1.0); opt.step()
                losses.append(loss.item())
            layer.eval()
            with torch.no_grad(): _, attF = layer(GEh, EI, return_attention=True)
            attF = attF.detach().cpu()

            # ----- outputs in the pipeline format -----
            gdir = OUT / f"seed_{seed}" / sid / f"{sid}_plot" / "grn"; gdir.mkdir(parents=True, exist_ok=True)
            with contextlib.redirect_stdout(io.StringIO()):
                run_global_grn_report(gene_graphs=gg, gene_names=list(gn_h), grn_attn_weights=attF,
                                      out_dir=str(gdir), prefix=sid, make_plots=(seed == MAKE_PLOTS_SEED),
                                      adata=ad_h, cluster_key="niche")
                niche_edge_activity(adata=ad_h, gene_graph=gg, gene_names=list(gn_h), attention=attF,
                                    cluster_key="niche").to_csv(gdir / "niche_edge_activity.csv")
            torch.save({"grn_layer": layer.state_dict(), "grn_expr_decoder": dec.state_dict(), "hvg": [names[i] for i in HV],
                        "attention": attF, "edge_index": EI.cpu()}, gdir / f"{sid}_grn_metacell.pt")
            LOG.append({"sample_id": sid, "seed": seed, "n_genes": Gh, "n_edges": EI.shape[1], "n_spearman": n_sp,
                        "self_loops_removed": n_self, "n_metacells": Z.shape[0],
                        "loss_first": np.mean(losses[:20]), "loss_last": np.mean(losses[-20:]),
                        "init_vs_final_spearman": spearmanr(a0.detach().cpu(), attF)[0]})
            print(f"{sid} seed {seed}: loss {LOG[-1]['loss_first']:.3f}->{LOG[-1]['loss_last']:.3f} | "
                  f"init~final {LOG[-1]['init_vs_final_spearman']:.3f}")
        print(f"   {sid}: {Gh} genes, {EI.shape[1]} edges ({n_sp} Spearman), {Z.shape[0]} metacells, "
              f"{(time.time() - t0) / 60:.1f} min")
        del Z; torch.cuda.empty_cache()
    except Exception as e:
        print(f"!! {sid} failed: {type(e).__name__}: {e}")
        LOG.append({"sample_id": sid, "error": f"{type(e).__name__}: {e}"})
    pd.DataFrame(LOG).to_csv(OUT / "run_log.csv", index=False)
print(f"\ndone -> {OUT}")

17 sections, seeds [1990], HVG mode per_section
Loading results_v2/pdac_k10/h5ad/2130785-C1.h5ad
Covered genes: 4988/4988 (100.00%)
  26463 cells, 4988 genes
  2130785-C1.pt: 26463 cells, already aligned
2130785-C1 seed 1990: loss 1.025->0.920 | init~final 0.604
   2130785-C1: 500 genes, 1756 edges (79 Spearman), 877 metacells, 0.4 min
Loading results_v2/pdac_k10/h5ad/2146816-C2.h5ad
Covered genes: 4988/4988 (100.00%)
  14442 cells, 4988 genes
  2146816-C2.pt: 14442 cells, already aligned
2146816-C2 seed 1990: loss 1.030->0.912 | init~final 0.575
   2146816-C2: 500 genes, 2041 edges (14 Spearman), 476 metacells, 0.3 min
Loading results_v2/pdac_k10/h5ad/2214414-C6.h5ad
Covered genes: 4988/4988 (100.00%)
  41045 cells, 4988 genes
  2214414-C6.pt: 41045 cells, already aligned
2214414-C6 seed 1990: loss 1.031->0.928 | init~final 0.549
   2214414-C6: 500 genes, 2208 edges (10 Spearman), 1362 metacells, 0.4 min
Loading results_v2/pdac_k10/h5ad/2334564-A4.h5ad
Covered genes: 4988/4988 (100.00

In [22]:
# %% GRN (metacell stage) report: seed 1990 = main result; reproducibility of R vs NR edges/hubs across 5 seeds
import glob, itertools, numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

ROOT = Path("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10")
GM = ROOT / "grn_metacell"
REP = GM / "report"; REP.mkdir(exist_ok=True)
MAIN, SEEDS = 1990, [1990, 11, 22, 33, 44]
OTHER = [s for s in SEEDS if s != MAIN]
ANN = pd.read_csv("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10/HEBI_pdac_k10_annotation_final_v3.csv")
meta = ANN.drop_duplicates("sample_id").set_index("sample_id")[["patient_id", "response"]]
meta["patient_id"] = meta.patient_id.astype(str)
MIN_OUT, TOPK = 5, 5

def gdir(seed, sid): return GM / f"seed_{seed}" / sid / f"{sid}_plot" / "grn"

# ---------- load relative attention (prior edges with >1 competing regulator) ----------
RAW = {}
for s in SEEDS:
    RAW[s] = {}
    for f in sorted(glob.glob(str(GM / f"seed_{s}" / "*" / "*_plot/grn/*_edges_ranked_competitive.csv"))):
        sid = Path(f).name.replace("_edges_ranked_competitive.csv", "")
        x = pd.read_csv(f, usecols=["source_gene", "target_gene", "attention", "evidence_tier", "target_in_degree"])
        x = x[(x.evidence_tier != "data_only") & (x.target_in_degree > 1)]
        x.index = x.source_gene + "->" + x.target_gene
        RAW[s][sid] = (x.attention * x.target_in_degree).groupby(level=0).mean()
secs = sorted(set.intersection(*[set(v) for v in RAW.values()]))
shared = sorted(set.intersection(*[set(e.index) for v in RAW.values() for e in v.values()]))
PCT = {s: pd.DataFrame({sid: RAW[s][sid].loc[shared].rank(pct=True) for sid in secs}) for s in SEEDS}
print(f"{len(secs)} sections in all seeds | {len(shared)} shared prior edges")

# ---------- 1. attention reproducibility + seed fingerprint ----------
rows = []
for sid in secs:
    for a, b in itertools.combinations(SEEDS, 2):
        pa, pb = PCT[a][sid], PCT[b][sid]; k = max(1, int(0.05 * len(pa)))
        rows.append({"sample_id": sid, "pair": f"{a}-{b}", "spearman": spearmanr(pa, pb)[0],
                     "top5pct_jaccard": len(set(pa.nlargest(k).index) & set(pb.nlargest(k).index)) /
                                        len(set(pa.nlargest(k).index) | set(pb.nlargest(k).index))})
rep = pd.DataFrame(rows); rep.to_csv(REP / "1_attention_reproducibility.csv", index=False)
same_seed = np.mean([spearmanr(PCT[s][a], PCT[s][b])[0] for s in SEEDS for a, b in itertools.combinations(secs, 2)])
diff_seed = np.mean([spearmanr(PCT[s1][a], PCT[s2][b])[0] for s1, s2 in itertools.combinations(SEEDS, 2)
                     for a, b in itertools.combinations(secs, 2)])
print(f"\n1. same section, different seeds: Spearman median {rep.spearman.median():.3f} "
      f"(min {rep.spearman.min():.3f}) | top-5% Jaccard median {rep.top5pct_jaccard.median():.3f}")
print(f"   different sections: same seed {same_seed:.3f} vs different seeds {diff_seed:.3f} (equal = no fingerprint)")

# ---------- 2. R vs NR per seed, patient level, exact permutation ----------
pid = meta.loc[secs, "patient_id"].values
pats = sorted(set(pid))
resp = meta.drop_duplicates("patient_id").set_index("patient_id").response.astype(str).loc[pats]
isR = (resp == "R").values
labs = np.array([np.isin(np.arange(len(pats)), c) for c in itertools.combinations(range(len(pats)), isR.sum())])
obs = np.where((labs == isR).all(axis=1))[0][0]
WR = (labs / labs.sum(1, keepdims=True)).T; WN = ((~labs) / (~labs).sum(1, keepdims=True)).T
src = pd.Series(shared).str.split("->").str[0]
hg = src.value_counts(); hg = hg[hg >= MIN_OUT].index
Mh = pd.get_dummies(src).loc[:, hg].values.astype(float); Mh /= Mh.sum(0, keepdims=True)
print(f"   patients: R {isR.sum()}, NR {(~isR).sum()} | hubs (>= {MIN_OUT} shared out-edges): {len(hg)}")

def ptest(V):
    v = V[:, obs]
    p = np.minimum(1, 2 * np.minimum((V >= v[:, None] - 1e-12).mean(1), (V <= v[:, None] + 1e-12).mean(1)))
    return v, p
RES, SGN = {}, {"edges": [], "hubs": []}
for s in SEEDS:
    P = PCT[s].T.groupby(pid).mean().T[pats].values                  # edges x patients
    D = P @ WR - P @ WN                                                # edges x 495 labelings
    H = Mh.T @ D
    de, pe = ptest(D); dh, ph = ptest(H)
    RES[s] = {"edges": pd.DataFrame({"delta": de, "p": pe}, index=shared),
              "hubs": pd.DataFrame({"delta": dh, "p": ph}, index=hg)}
    SGN["edges"].append(np.sign(D).astype(np.int8)); SGN["hubs"].append(np.sign(H).astype(np.int8))

# ---------- 3. main table (1990) + reproducibility in the other seeds ----------
print()
for lvl in ["edges", "hubs"]:
    T = RES[MAIN][lvl].copy()
    T["direction"] = np.where(T.delta > 0, "R", "NR")
    T["padj_BH"] = multipletests(T.p, method="fdr_bh")[1]
    others_d = pd.DataFrame({s: RES[s][lvl].delta for s in OTHER})
    others_p = pd.DataFrame({s: RES[s][lvl].p for s in OTHER})
    T["n_other_seeds_same_sign"] = (np.sign(others_d).values == np.sign(T.delta.values)[:, None]).sum(1)
    T["n_other_seeds_same_sign_p<0.05"] = ((np.sign(others_d).values == np.sign(T.delta.values)[:, None]) &
                                           (others_p.values < 0.05)).sum(1)
    T["delta_mean_5_seeds"] = pd.DataFrame({s: RES[s][lvl].delta for s in SEEDS}).mean(1)
    T = T.sort_values("p"); T.to_csv(REP / f"3_{lvl}_R_vs_NR_seed{MAIN}.csv")
    cor = pd.DataFrame({s: RES[s][lvl].delta for s in SEEDS}).corr(method="spearman").loc[MAIN, OTHER]
    sig = T[T.p < 0.05]
    print(f"3. {lvl}: R-NR delta, Spearman 1990 vs other seeds {cor.round(2).tolist()}")
    print(f"   p<0.05 in 1990: {len(sig)} (R {int((sig.direction == 'R').sum())}, NR {int((sig.direction == 'NR').sum())}); "
          f"of these, same sign in all 4 other seeds: {int((sig.n_other_seeds_same_sign == 4).sum())}, "
          f"also p<0.05 in >= 3 other seeds: {int((sig['n_other_seeds_same_sign_p<0.05'] >= 3).sum())}")
    if lvl == "hubs":
        for dr in ["R", "NR"]:
            x = sig[sig.direction == dr].head(20)
            print(f"   {dr} hubs (1990 p<0.05): " + ", ".join(
                f"{g}[{int(r['n_other_seeds_same_sign'])}/{int(r['n_other_seeds_same_sign_p<0.05'])}]" for g, r in x.iterrows()))
        print("   [a/b] = other seeds (of 4) with same sign / same sign and p<0.05")

# ---------- 4. all-5-seed sign agreement vs label permutation (labels shared across seeds) ----------
perm_rows = []
for lvl in ["edges", "hubs"]:
    S = np.stack(SGN[lvl]); cnt = ((S > 0).all(0) | (S < 0).all(0)).sum(0)
    perm_rows.append({"level": lvl, "n": S.shape[1], "observed": int(cnt[obs]), "perm_median": float(np.median(cnt)),
                      "perm_95pct": float(np.percentile(cnt, 95)), "perm_p": float((cnt >= cnt[obs]).mean())})
perm = pd.DataFrame(perm_rows); perm.to_csv(REP / "4_sign_agreement_permutation.csv", index=False)
print("\n4. same R/NR sign in all 5 seeds vs 495 label permutations:"); print(perm.round(3).to_string(index=False))

# ---------- 5. top-5 interactions per niche: stable across seeds? ----------
def top_set(seed, sid):
    f = gdir(seed, sid) / "niche_edge_activity.csv"
    if not f.exists(): return None
    A = pd.read_csv(f, index_col=0); A.index = A.index.str.replace(" → ", "->").str.replace(" ", "")
    return set().union(*[set(A[c].nlargest(TOPK).index) for c in A.columns])
tk = []
for sid in secs:
    T = {s: top_set(s, sid) for s in SEEDS}; T = {s: v for s, v in T.items() if v}
    tk.append({"sample_id": sid, "response": meta.loc[sid, "response"],
               "jaccard_1990_vs_others": np.mean([len(T[MAIN] & T[s]) / len(T[MAIN] | T[s]) for s in OTHER if s in T]),
               "n_top_1990": len(T[MAIN]),
               "n_top_1990_in_all_seeds": len(set.intersection(*T.values()))})
tk = pd.DataFrame(tk); tk.to_csv(REP / "5_top5_per_niche_reproducibility.csv", index=False)
print(f"\n5. top-{TOPK} per niche, Jaccard 1990 vs other seeds: median {tk.jaccard_1990_vs_others.median():.3f}; "
      f"fraction of 1990 top edges found in all seeds: {(tk.n_top_1990_in_all_seeds / tk.n_top_1990).median():.2f}")
print(f"\nCSVs -> {REP}")

17 sections in all seeds | 984 shared prior edges

1. same section, different seeds: Spearman median 0.812 (min 0.658) | top-5% Jaccard median 0.607
   different sections: same seed 0.256 vs different seeds 0.246 (equal = no fingerprint)
   patients: R 4, NR 8 | hubs (>= 5 shared out-edges): 63

3. edges: R-NR delta, Spearman 1990 vs other seeds [0.76, 0.75, 0.74, 0.75]
   p<0.05 in 1990: 67 (R 36, NR 31); of these, same sign in all 4 other seeds: 60, also p<0.05 in >= 3 other seeds: 26
3. hubs: R-NR delta, Spearman 1990 vs other seeds [0.79, 0.85, 0.85, 0.9]
   p<0.05 in 1990: 5 (R 3, NR 2); of these, same sign in all 4 other seeds: 5, also p<0.05 in >= 3 other seeds: 1
   R hubs (1990 p<0.05): PTPN1[4/3], IL2RG[4/1], MDM2[4/0]
   NR hubs (1990 p<0.05): CTNNB1[4/0], AKT3[4/2]
   [a/b] = other seeds (of 4) with same sign / same sign and p<0.05

4. same R/NR sign in all 5 seeds vs 495 label permutations:
level   n  observed  perm_median  perm_95pct  perm_p
edges 984       569        553

In [31]:
# %% PDAC GRN (metacell, seed 1990): what is shared across patients? interactions, hubs, programmes + 3 plots
import glob, time, numpy as np, pandas as pd, gseapy as gp, matplotlib as mpl, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10")
GM = ROOT / "grn_metacell_persection" / "seed_1990"
OUTD = ROOT / "grn_metacell_persection" / "shared_across_patients"; OUTD.mkdir(exist_ok=True)
ANN = pd.read_csv("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10/HEBI_pdac_k10_annotation_final_v3.csv")
meta = ANN.drop_duplicates("sample_id").set_index("sample_id")[["patient_id", "response"]]
meta["patient_id"] = meta.patient_id.astype(str)
TOPK, TOP_HUBS, SHARED = 5, 10, 0.5            # top-5 per niche, top-10 hubs per section, shared = >= 50% of patients
LIBS = ["Reactome_Pathways_2024", "GO_Biological_Process_2023"]
R_COL, NR_COL = "#4A3B75", "#B31B54"
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 9, "svg.fonttype": "path"})
def save(fig, name):
    fig.savefig(OUTD / f"{name}.svg", bbox_inches="tight")
    fig.savefig(OUTD / f"{name}.png", dpi=600, bbox_inches="tight", facecolor="white"); plt.close(fig)

# ---------- 1. per section: top-5 interactions per niche, top hubs ----------
INT, HUBS, GENES_ALL = {}, {}, set()
for d in sorted(GM.iterdir()):
    sid = d.name; g = d / f"{sid}_plot" / "grn"
    if not (g / "niche_edge_activity.csv").exists(): continue
    A = pd.read_csv(g / "niche_edge_activity.csv", index_col=0)
    A.index = A.index.str.replace(" → ", "->").str.replace(" ", "")
    INT[sid] = set().union(*[set(A[c].nlargest(TOPK).index) for c in A.columns])
    e = pd.read_csv(g / f"{sid}_edges_ranked_competitive.csv", usecols=["source_gene", "target_gene", "attention", "target_in_degree"])
    e["rel"] = e.attention * e.target_in_degree
    strong = e[e.rel.rank(pct=True) >= 0.90]
    HUBS[sid] = set(strong.source_gene.value_counts().head(TOP_HUBS).index)
    GENES_ALL |= set(e.source_gene) | set(e.target_gene)
secs = sorted(INT)
pid = meta.loc[secs, "patient_id"]
pats_R = sorted(set(pid[meta.loc[secs, "response"] == "R"])); pats_NR = sorted(set(pid[meta.loc[secs, "response"] == "NR"]))
pats = pats_R + pats_NR
print(f"{len(secs)} sections, {len(pats)} patients (R {len(pats_R)}, NR {len(pats_NR)})")

def by_patient(D):
    """item x patient presence (present in any section of that patient)"""
    items = sorted(set().union(*D.values()))
    M = pd.DataFrame({s: [i in D[s] for i in items] for s in D}, index=items)
    return M.T.groupby(pid.loc[M.columns].values).any().T[pats].astype(int)
IP, HP = by_patient(INT), by_patient(HUBS)
for name, M in [("interactions", IP), ("hubs", HP)]:
    f = M.mean(1).sort_values(ascending=False)
    out = pd.DataFrame({"frac_patients": f, "n_R": M.loc[f.index, pats_R].sum(1), "n_NR": M.loc[f.index, pats_NR].sum(1)})
    out.to_csv(OUTD / f"{name}_by_patient.csv")
    print(f"\n{name}: {len(M)} total | shared (>= {SHARED:.0%} of patients): {(f >= SHARED).sum()}")
    print(out.head(15).to_string(float_format=lambda v: f"{v:.2f}"))

# ---------- 2. programmes of the shared interactions ----------
sh_int = IP.index[IP.mean(1) >= SHARED]
prog = sorted({g for e in sh_int for g in e.split("->")})
if "GMT" not in globals() or not all(l in GMT for l in LIBS):
    GMT = {}
    for lib in LIBS:
        for attempt in range(5):
            try: GMT[lib] = gp.get_library(name=lib, organism="Human"); break
            except Exception as exc: print(f"  {lib}: {exc}"); time.sleep(10)
enr = pd.concat([gp.enrich(gene_list=prog, gene_sets=gs, background=sorted(GENES_ALL), outdir=None, no_plot=True)
                 .results.assign(library=lib) for lib, gs in GMT.items()], ignore_index=True)
enr["Term"] = enr.Term.str.replace(r"\s*\(GO:\d+\)", "", regex=True).str.replace(r"\s*R-HSA-\d+", "", regex=True)
enr["set_size"] = enr.Overlap.str.split("/").str[1].astype(int)
enr = enr[enr.set_size <= 40].sort_values("P-value"); enr.to_csv(OUTD / "shared_programmes_enrichment.csv", index=False)
print(f"\nshared-interaction genes: {len(prog)} | background {len(GENES_ALL)}")
print(enr.head(10)[["library", "Term", "Overlap", "P-value", "Adjusted P-value"]].to_string(index=False))

# ---------- 3. plots ----------
def presence_plot(M, n, title, name):
    M = M.loc[M.mean(1).sort_values(ascending=False).index[:n]]
    fig, ax = plt.subplots(figsize=(0.32 * len(pats) + 2.2, 0.22 * len(M) + 0.9))
    cm = mpl.colors.ListedColormap(["#f2f2f2", "#555555"])
    ax.imshow(M.values, aspect="auto", cmap=cm, vmin=0, vmax=1)
    for j, p in enumerate(pats):
        ax.add_patch(plt.Rectangle((j - 0.5, -1.4), 1, 0.7, color=R_COL if p in pats_R else NR_COL, clip_on=False))
    ax.set_xticks(range(len(pats)), pats, rotation=90, fontsize=7)
    ax.set_yticks(range(len(M)), M.index, fontsize=8, style="italic")
    ax.set_xticks(np.arange(-0.5, len(pats)), minor=True); ax.set_yticks(np.arange(-0.5, len(M)), minor=True)
    ax.grid(which="minor", color="white", lw=0.8); ax.tick_params(which="both", length=0)
    ax.axvline(len(pats_R) - 0.5, color="k", lw=1)
    ax.set_ylim(len(M) - 0.5, -1.5); ax.set_title(title, loc="left", fontsize=10, pad=12)
    for s_ in ax.spines.values(): s_.set_visible(False)
    save(fig, name)
presence_plot(IP, 25, "Top niche interactions, presence per patient (R | NR)", "shared_interactions_by_patient")
presence_plot(HP, 20, "Hub genes, presence per patient (R | NR)", "shared_hubs_by_patient")

t = enr.head(8).iloc[::-1]
fig, ax = plt.subplots(figsize=(5, 2.6))
v = -np.log10(t["P-value"].values); y = np.arange(len(t))
ax.hlines(y, 0, v, color="#444444", lw=2); ax.scatter(v, y, color="#444444", s=30, zorder=3)
for yi, vi, ov in zip(y, v, t.Overlap): ax.text(vi * 1.03, yi, ov, va="center", fontsize=7)
ax.set_yticks(y, [s if len(s) < 45 else s[:42] + "..." for s in t.Term], fontsize=8)
ax.axvline(-np.log10(0.05), color="#999999", ls="--", lw=0.8); ax.set_xlim(0, v.max() * 1.3)
ax.set_xlabel(r"$-\log_{10}$ $P$"); ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Programmes of interactions shared across patients", loc="left", fontsize=10)
save(fig, "shared_programmes")
print(f"\nplots + CSVs -> {OUTD}")

17 sections, 12 patients (R 4, NR 8)

interactions: 147 total | shared (>= 50% of patients): 3
                 frac_patients  n_R  n_NR
ATM->EEF1G                0.67    4     4
CCN1->CCN2                0.67    3     5
THBS1->CCN2               0.50    3     3
CCN1->NR4A1               0.42    1     4
SFPQ->NONO                0.42    2     3
THBS1->SERPINE1           0.42    4     1
COL5A1->COL5A2            0.33    0     4
CCN1->THBS1               0.33    1     3
ATP1A1->CFTR              0.33    2     2
STAT1->PLEC               0.33    0     4
HIF1A->ERRFI1             0.33    1     3
ANXA5->ANXA2              0.33    0     4
RAC1->ACTN4               0.33    0     4
CCN2->COL5A1              0.25    0     3
COL5A1->THY1              0.25    0     3

hubs: 72 total | shared (>= 50% of patients): 5
        frac_patients  n_R  n_NR
RAC1             0.67    3     5
ETS1             0.58    3     4
PDGFRA           0.58    2     5
CCN1             0.50    2     4
MYC              0.

In [33]:
# ---------- 4. programmes per patient (genes of that patient's top interactions), then shared terms ----------
pat_genes = {}
for p in pats:
    ss = [s for s in secs if pid[s] == p]
    pat_genes[p] = sorted({g for s in ss for e in INT[s] for g in e.split("->")})
rows = []
for p, genes in pat_genes.items():
    for lib, gs in GMT.items():
        r = gp.enrich(gene_list=genes, gene_sets=gs, background=sorted(GENES_ALL), outdir=None, no_plot=True).results
        rows.append(r.assign(patient=p, library=lib))
pe = pd.concat(rows, ignore_index=True)
pe["Term"] = pe.Term.str.replace(r"\s*\(GO:\d+\)", "", regex=True).str.replace(r"\s*R-HSA-\d+", "", regex=True)
pe["set_size"] = pe.Overlap.str.split("/").str[1].astype(int)
pe["n_hit"] = pe.Overlap.str.split("/").str[0].astype(int)
pe = pe[(pe.set_size <= 40) & (pe.n_hit >= 2)]                     # >= 2 genes, not too broad
pe.to_csv(OUTD / "programmes_per_patient.csv", index=False)
TP = (pe[pe["P-value"] < 0.05].assign(v=1).pivot_table(index="Term", columns="patient", values="v", aggfunc="max")
      .reindex(columns=pats).fillna(0).astype(int))
f = TP.mean(1).sort_values(ascending=False)
summ = pd.DataFrame({"frac_patients": f, "n_R": TP.loc[f.index, pats_R].sum(1), "n_NR": TP.loc[f.index, pats_NR].sum(1)})
summ.to_csv(OUTD / "programmes_shared_across_patients.csv")
print(f"\ngenes per patient: " + ", ".join(f"{p} {len(g)}" for p, g in pat_genes.items()))
print(f"programmes (P<0.05, >= 2 genes) in >= {SHARED:.0%} of patients: {(f >= SHARED).sum()}")
print(summ.head(25).to_string(float_format=lambda v: f"{v:.2f}"))
presence_plot(TP, 25, "Programmes (P < 0.05) per patient (R | NR)", "shared_programmes_by_patient")


genes per patient: 965740 18, AAB746 15, AAY085 17, D28560 31, AAO748 33, ABV534 30, BBA636 18, BBC829 48, BBT131 19, BCA213 41, D49486 20, D93346 39
programmes (P<0.05, >= 2 genes) in >= 50% of patients: 17
                                                                                frac_patients  n_R  n_NR
Term                                                                                                    
Negative Regulation Of Angiogenesis                                                      0.67    3     5
Regulation Of Angiogenesis                                                               0.58    3     4
Regulation Of Plasminogen Activation                                                     0.50    4     2
Apoptotic Cell Clearance                                                                 0.50    1     5
Negative Regulation Of Fibrinolysis                                                      0.50    4     2
Negative Regulation Of Endothelial Cell Proliferation   

In [35]:
# %% per patient: top-5 Spearman (data_only) and top-5 prior_only interactions by learned attention; heatmap + print
import glob, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt
from pathlib import Path

ROOT = Path("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10")
GM = ROOT / "grn_metacell_persection" / "seed_1990"
OUTD = ROOT / "grn_metacell_persection" / "shared_across_patients"; OUTD.mkdir(exist_ok=True)
ANN = pd.read_csv("/DATA1/Roxana/hebi_repo/notebooks/results_v2/pdac_k10/HEBI_pdac_k10_annotation_final_v3.csv")
meta = ANN.drop_duplicates("sample_id").set_index("sample_id")[["patient_id", "response"]]
meta["patient_id"] = meta.patient_id.astype(str)
TOPN = 5
VALUE = "pct"                 # "pct" = within-section percentile of attention x in-degree; "rel" = attention x in-degree
R_COL, NR_COL = "#4A3B75", "#B31B54"
mpl.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8, "svg.fonttype": "path"})

# ---------- load: per section, competitive edges with tier and learned attention ----------
rows = []
for f in sorted(glob.glob(str(GM / "*" / "*_plot/grn/*_edges_ranked_competitive.csv"))):
    sid = Path(f).name.replace("_edges_ranked_competitive.csv", "")
    d = pd.read_csv(f, usecols=["source_gene", "target_gene", "attention", "evidence_tier", "target_in_degree"])
    d["edge"] = d.source_gene + "->" + d.target_gene
    d = d.drop_duplicates("edge")
    d["rel"] = d.attention * d.target_in_degree
    d["pct"] = d.rel.rank(pct=True)
    rows.append(d.assign(sample_id=sid, patient=meta.loc[sid, "patient_id"], response=meta.loc[sid, "response"]))
E = pd.concat(rows, ignore_index=True)
E = E[E.evidence_tier.isin(["data_only", "prior_only"])]
pat_resp = E.drop_duplicates("patient").set_index("patient").response
pats = sorted(pat_resp[pat_resp == "R"].index) + sorted(pat_resp[pat_resp == "NR"].index)
nR = int((pat_resp == "R").sum())

# patient value = mean over that patient's sections that contain the edge
PV = E.groupby(["evidence_tier", "edge", "patient"])[[VALUE, "attention", "rel"]].mean()

# ---------- top-5 per patient per tier, print ----------
TOP = {t: {} for t in ["data_only", "prior_only"]}
for t in TOP:
    for p in pats:
        x = PV.loc[t].xs(p, level="patient").sort_values(VALUE, ascending=False).head(TOPN)
        TOP[t][p] = list(x.index)
        print(f"{'Spearman' if t == 'data_only' else 'prior'} | {pat_resp[p]} {p} ({(E.patient == p).groupby(E.sample_id).any().sum()} sec): "
              + ", ".join(f"{e} ({r.attention:.2f}, rel {r.rel:.2f})" for e, r in x.iterrows()))
    print()
out = pd.DataFrame([{"tier": t, "patient": p, "response": pat_resp[p], "rank": i + 1, "edge": e,
                     **PV.loc[(t, e, p)].to_dict()} for t in TOP for p in pats for i, e in enumerate(TOP[t][p])])
out.to_csv(OUTD / "top5_spearman_prior_per_patient.csv", index=False)

# ---------- heatmap: rows = union of top edges (ordered by how many patients have them in their top-5) ----------
blocks = []
for t in ["data_only", "prior_only"]:
    cnt = pd.Series([e for p in pats for e in TOP[t][p]]).value_counts()
    first = {e: min(pats.index(p) for p in pats if e in TOP[t][p]) for e in cnt.index}
    order = sorted(cnt.index, key=lambda e: (-cnt[e], first[e]))
    M = PV.loc[t][VALUE].unstack("patient").reindex(index=order, columns=pats)
    T = pd.DataFrame([[e in TOP[t][p] for p in pats] for e in order], index=order, columns=pats)
    blocks.append((t, M, T, cnt))
h = [len(b[1]) for b in blocks]
fig, axes = plt.subplots(2, 1, figsize=(0.32 * len(pats) + 3.2, 0.16 * sum(h) + 1.6),
                         gridspec_kw={"height_ratios": h, "hspace": 0.08})
cmap = mpl.cm.get_cmap("RdPu").copy(); cmap.set_bad("#e6e6e6")
vmin, vmax = (0.5, 1.0) if VALUE == "pct" else (np.nanpercentile(PV[VALUE], 5), np.nanpercentile(PV[VALUE], 99))
for k, (ax, (t, M, T, cnt)) in enumerate(zip(axes, blocks)):
    im = ax.imshow(np.ma.masked_invalid(M.values), aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)
    yy, xx = np.where(T.values); ax.scatter(xx, yy, s=6, c="white", edgecolor="k", linewidths=0.3, zorder=3)
    ax.set_yticks(range(len(M)), [f"{e} ({cnt[e]})" for e in M.index], fontsize=6, style="italic")
    ax.set_xticks(np.arange(-0.5, len(pats)), minor=True); ax.set_yticks(np.arange(-0.5, len(M)), minor=True)
    ax.grid(which="minor", color="white", lw=0.5); ax.tick_params(which="both", length=0)
    ax.axvline(nR - 0.5, color="k", lw=1)
    ax.set_ylabel("Spearman (data-driven)" if t == "data_only" else "Prior (curated)", fontsize=8)
    for s_ in ax.spines.values(): s_.set_visible(False)
    if k == 0:
        for j, p in enumerate(pats):
            ax.add_patch(plt.Rectangle((j - 0.5, -1.6), 1, 0.9, color=R_COL if pat_resp[p] == "R" else NR_COL, clip_on=False))
        ax.set_ylim(len(M) - 0.5, -1.7); ax.set_xticks([])
    else:
        ax.set_xticks(range(len(pats)), pats, rotation=90, fontsize=7)
cb = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02)
cb.set_label("Learned attention (within-section percentile)" if VALUE == "pct" else "Relative attention (attention x in-degree)")
cb.solids.set_rasterized(False)
fig.suptitle(f"Top-{TOPN} learned interactions per patient (R | NR); dot = in that patient's top {TOPN}; grey = edge not in graph",
             x=0.02, ha="left", fontsize=9)
for ext, kw in [("svg", {}), ("png", {"dpi": 600, "facecolor": "white"})]:
    fig.savefig(OUTD / f"top5_spearman_prior_per_patient.{ext}", bbox_inches="tight", **kw)
plt.close(fig)
print(f"heatmap + CSV -> {OUTD}")

Spearman | R 965740 (1 sec): NUCB2->XBP1 (0.56, rel 5.03), SEL1L->XBP1 (0.44, rel 3.92), CCN1->CCN2 (0.55, rel 3.88), ERP27->TMEM97 (0.39, rel 3.52), ANPEP->PDIA2 (0.36, rel 2.92)
Spearman | R AAB746 (1 sec): CCN1->THBS1 (0.52, rel 5.68), CCN1->CCN2 (0.71, rel 5.66)
Spearman | R AAY085 (2 sec): CD3E->CXCR4 (0.52, rel 8.86), PLVAP->S1PR1 (0.72, rel 9.35), CFTR->CTNND1 (0.69, rel 7.60), ADIPOQ->LPL (0.83, rel 4.96), CCN1->CCN2 (0.53, rel 4.84)
Spearman | R D28560 (1 sec): CLEC4M->CXCL2 (0.94, rel 2.82), CD19->MS4A1 (0.43, rel 1.72), MS4A1->TNFRSF13C (0.74, rel 1.47), PECAM1->SHANK3 (0.71, rel 1.41), BANK1->MS4A1 (0.27, rel 1.06)
Spearman | NR AAO748 (1 sec): CCN1->NR4A1 (1.00, rel 5.98), CCN1->CCN2 (0.73, rel 5.11), CTNND1->PLAT (0.35, rel 3.88), SLCO2B1->STAB1 (0.51, rel 3.04), COL5A1->COL5A2 (0.93, rel 2.78)
Spearman | NR ABV534 (1 sec): SEL1L->XBP1 (0.64, rel 10.83), CUZD1->PDCD4 (0.74, rel 6.69), CFTR->SERPINA3 (0.45, rel 6.24), COL5A1->THBS2 (0.76, rel 5.31), COL5A1->COL5A2 (0.86, r